In [34]:
import re
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectPercentile, chi2
from sklearn.metrics import classification_report

In [4]:
DATA_PATH = "D:/NLP/Job/final_project.ods"

data = pd.read_excel(DATA_PATH, engine="odf", dtype=str)

print(f" Kích thước ban đầu: {data.shape}")
data.head()

 Kích thước ban đầu: (8074, 6)


,title,location,description,function,industry,career_level
0,Technical Professional Lead - Process,"Houston, TX","Responsible for the study, design, and specifi...",production_manufacturing,Machinery and Industrial Facilities Engineering,senior_specialist_or_project_manager
1,Cnslt - Systems Eng- Midrange 1,"Seattle, WA","Participates in design, development and implem...",information_technology_telecommunications,Financial Services,senior_specialist_or_project_manager
2,SharePoint Developers and Solution Architects,"Dallas, TX",We are currently in need of Developers who can...,consulting,IT Consulting,senior_specialist_or_project_manager
3,Business Information Services - Strategic Acco...,North Carolina,Experian is seeking an experienced Account Exe...,sales,"Security, Risk, Restructuring Consulting",senior_specialist_or_project_manager
4,Strategic Development Director (procurement),"Austin, TX",Â Want to join a world-class global procuremen...,procurement_materials_logistics,Information Technology,bereichsleiter


##  Làm sạch dữ liệu
- Xóa các hàng có giá trị `NaN`
- Chuẩn hóa cột `location`: chỉ giữ mã bang 2 chữ hoa nếu có (ví dụ: `"New York, NY"` → `"NY"`)

In [7]:
def filter_location(location: str) -> str:
    """
    Trích xuất mã bang 2 chữ hoa ở cuối chuỗi location.
    Ví dụ: 'San Francisco, CA' → 'CA'
    Nếu không khớp định dạng, giữ nguyên chuỗi gốc.
    """
    result = re.findall(r",\s[A-Z]{2}$", location)
    if result:
        return result[0][2:]   
    return location


data = data.dropna(axis=0)

data["location"] = data["location"].apply(filter_location)

print(f" Kích thước sau làm sạch: {data.shape}")
print("\nPhân phối nhãn career_level:")
print(data["career_level"].value_counts())

 Kích thước sau làm sạch: (8073, 6)

Phân phối nhãn career_level:
career_level
senior_specialist_or_project_manager      4337
manager_team_leader                       2672
bereichsleiter                             960
director_business_unit_leader               70
specialist                                  30
managing_director_small_medium_company       4
Name: count, dtype: int64


##  Tách features và nhãn, chia tập train/test
- `stratify=y`: đảm bảo tỷ lệ nhãn trong train/test giống nhau
- `random_state=100`: cố định kết quả tái hiện

In [23]:
TARGET = "career_level"

X = data.drop(TARGET, axis=1)  # Ma trận đặc trưng
y = data[TARGET]               # Nhãn mục tiêu

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=100,
    stratify=y           
)

print(f"Train size : {X_train.shape[0]} mẫu")
print(f"Test size  : {X_test.shape[0]} mẫu")
print("\nPhân phối nhãn trong tập train:")
print(y_train.value_counts())

Train size : 6458 mẫu
Test size  : 1615 mẫu

Phân phối nhãn trong tập train:
career_level
senior_specialist_or_project_manager      3469
manager_team_leader                       2138
bereichsleiter                             768
director_business_unit_leader               56
specialist                                  24
managing_director_small_medium_company       3
Name: count, dtype: int64


## Định nghĩa Preprocessor
Dùng `ColumnTransformer` để xử lý song song từng loại cột:
| Cột | Phương pháp | Ghi chú |
|---|---|---|
| `title` | TF-IDF unigram | Tên vị trí tuyển dụng |
| `location` | OneHotEncoder | Mã bang / địa điểm |
| `description` | TF-IDF bigram | Mô tả công việc |
| `function` | OneHotEncoder | Chức năng bộ phận |
| `industry` | TF-IDF unigram | Ngành nghề |

In [18]:
preprocessor = ColumnTransformer(transformers=[
    (
        "title_tfidf",
        TfidfVectorizer(stop_words="english", ngram_range=(1, 1)),
        "title"           
    ),
    (
        "location_ohe",
        OneHotEncoder(handle_unknown="ignore"),  
        ["location"]       
    ),
    (
        "description_tfidf",
        TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),   
            min_df=0.01,          
            max_df=0.95           
        ),
        "description"
    ),
    (
        "function_ohe",
        OneHotEncoder(handle_unknown="ignore"),
        ["function"]
    ),
    (
        "industry_tfidf",
        TfidfVectorizer(stop_words="english", ngram_range=(1, 1)),
        "industry"
    ),
])

print("Preprocessor đã được định nghĩa")

Preprocessor đã được định nghĩa


##  Xây dựng Pipeline
Pipeline gồm 3 bước nối tiếp nhau:
1. **preprocessor** — biến đổi raw text/categorical → ma trận số
2. **feature_selector** — chọn top `percentile`% đặc trưng theo Chi-squared
3. **model** — Random Forest classifier

In [20]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    (
        "feature_selector",
        SelectPercentile(chi2, percentile=5)  
    ),
    (
        "model",
        RandomForestClassifier(random_state=42)
    ),
])

print(" Pipeline đã được xây dựng")
print(pipeline)

 Pipeline đã được xây dựng
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('title_tfidf',
                                                  TfidfVectorizer(stop_words='english'),
                                                  'title'),
                                                 ('location_ohe',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['location']),
                                                 ('description_tfidf',
                                                  TfidfVectorizer(max_df=0.95,
                                                                  min_df=0.01,
                                                                  ngram_range=(1,
                                                                               2),
                                                                  stop_words='english'),
                      

 GridSearchCV: tìm siêu tham số tốt nhất
- **`criterion`**: tiêu chí chia nhánh cây quyết định (`gini`, `entropy`, `log_loss`)
- **`feature_selector__percentile`**: thử các mức chọn đặc trưng 1%, 5%, 10%
- **`scoring="recall_weighted"`**: tối ưu recall có trọng số (phù hợp dữ liệu mất cân bằng)
- **`cv=4`**: 4-fold cross-validation

In [ ]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=4,
    scoring="recall_weighted",
    verbose=2,
    n_jobs=1   
)

In [31]:
print("Bắt đầu GridSearchCV...")
grid_search.fit(X_train, y_train)
print(f"Tham số tốt nhất : {grid_search.best_params_}")
print(f"Recall tốt nhất  : {grid_search.best_score_:.4f}")

Bắt đầu GridSearchCV...
Fitting 4 folds for each of 9 candidates, totalling 36 fits


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:725: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=4.
  warnings.warn(


[CV] END feature_selector__percentile=1, model__criterion=gini; total time=  10.8s
[CV] END feature_selector__percentile=1, model__criterion=gini; total time=   6.4s
[CV] END feature_selector__percentile=1, model__criterion=gini; total time=   6.2s
[CV] END feature_selector__percentile=1, model__criterion=gini; total time=   6.0s
[CV] END feature_selector__percentile=1, model__criterion=entropy; total time=   6.4s
[CV] END feature_selector__percentile=1, model__criterion=entropy; total time=   6.3s
[CV] END feature_selector__percentile=1, model__criterion=entropy; total time=   6.2s
[CV] END feature_selector__percentile=1, model__criterion=entropy; total time=   6.1s
[CV] END feature_selector__percentile=1, model__criterion=log_loss; total time=   6.1s
[CV] END feature_selector__percentile=1, model__criterion=log_loss; total time=   6.6s
[CV] END feature_selector__percentile=1, model__criterion=log_loss; total time=   6.7s
[CV] END feature_selector__percentile=1, model__criterion=log_l

In [32]:
y_predicted = grid_search.predict(X_test)

print("Classification Report:")
print("=" * 60)
print(classification_report(y_test, y_predicted))

Classification Report:
                                        precision    recall  f1-score   support

                        bereichsleiter       0.62      0.12      0.20       192
         director_business_unit_leader       1.00      0.07      0.13        14
                   manager_team_leader       0.64      0.75      0.69       534
managing_director_small_medium_company       0.00      0.00      0.00         1
  senior_specialist_or_project_manager       0.83      0.92      0.87       868
                            specialist       0.00      0.00      0.00         6

                              accuracy                           0.76      1615
                             macro avg       0.52      0.31      0.32      1615
                          weighted avg       0.74      0.76      0.72      1615



c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [33]:
results_df = pd.DataFrame(grid_search.cv_results_)

cols = [
    "param_model__criterion",
    "param_feature_selector__percentile",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]

results_df[cols].sort_values("rank_test_score")

,param_model__criterion,param_feature_selector__percentile,mean_test_score,std_test_score,rank_test_score
3,gini,5,0.756117,0.002236,1
7,entropy,10,0.755344,0.008392,2
8,log_loss,10,0.755344,0.008392,2
6,gini,10,0.753795,0.004799,4
4,entropy,5,0.753020,0.003272,5
5,log_loss,5,0.753020,0.003272,5
1,entropy,1,0.634564,0.011658,7
2,log_loss,1,0.634564,0.011658,7
0,gini,1,0.634564,0.012957,9
